# Preprocessing

Imports and system path

In [1]:
import multiprocessing as mp
import os
import sys
from functools import partial
from itertools import chain

import numpy as np
import pandas as pd
import trimesh
from numba import njit
from scipy.spatial import KDTree
from tqdm import tqdm

# Go up THREE levels (project root directory)
project_root = os.path.dirname(os.path.dirname(os.getcwd()))
# Append the new path to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
    print("Project root added to sys.path")
else:
    print("Project root already in sys.path")

Project root added to sys.path


Processing `npt-HK4.gro` file into pandas dataframe

In [2]:
from utils import gro_processing as gp

# Data directory can be accessed due to root PATH we set previously
path = 'data/npt-HK4.gro'
file = os.path.join(project_root, path)

# Extracts data from .gro file into multi-index DataFrame (unsorted)
df_gro, title, num_atoms, box_dimensions = gp.read_gro(file, multiply=10, positions=True, velocities=False) # convert nm to Å  
    
# Checking
# df_gro
# molecules # display 

# Generating Molecule Meshes

Create `mol_meshes` dictionary, which contains 1501 individual molecule mesh. (i.e. `{1: mesh1, 2: mesh2, ... 1501: mesh1501}`)

In [3]:
from utils.generate_mol_meshes import molecules_to_meshes

mol_meshes = molecules_to_meshes(df_gro, box_dimensions, num_processes=None, context='fork')
print(len(mol_meshes))        # 1501
print(mol_meshes[1])          # trimesh.Trimesh object

Processing 1501 molecules with 8 logical cores: 100%|██████████| 1501/1501 [00:54<00:00, 27.79it/s]


1501
<trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>


# Export meshes

Users have two options to save `.npz` file or `.ply` file.
- `.npz` is much smaller $\sim400\,\mathrm{MB}$, but exports longer $\sim50\,\mathrm{s}$
- `.ply` file is larger $\sim1.00\,\mathrm{GB}$, but exports faster $\sim20\,\mathrm{s}$

In [ ]:
from utils.generate_mol_meshes import export_meshes

# Export to either npz file or ply directory
export_meshes(mol_meshes, path=path, export_format='ply', num_processes=None, context='fork')

Exporting 1501 meshes with 8 logical cores: 100%|██████████| 1501/1501 [00:09<00:00, 164.60it/s]


Successfully saved all meshes into directory: npt-HK4_meshes
File size: 1087.88 MB


# Import meshes (TODO)

Users have two options to save `.npz` file or `.ply` file.
- `.npz` slower import $\sim20\,\mathrm{s}$ (parallel does not work well here) 
- `.ply` fast import $\sim8\,\mathrm{s}$

In [ ]:
from utils.generate_mol_meshes import import_meshes

# Import either npz files or ply directory (auto-detect)
mol_meshes = import_meshes(path='npt-HK4_meshes.npz', num_processes=None, context='fork')
# mol_meshes = import_meshes(path='npt-HK4_meshes', num_processes=None, context='fork')

Loading 1498 meshes with 8 cores: 100%|██████████| 1498/1498 [00:08<00:00, 177.44it/s]


In [ ]:
import numpy as np
import trimesh

def npz_load_meshes(file):
    """
    Extracts all mesh components from .npz, (e.g. 'npt-HK4_meshes.npz'), 
    then reconstructs trimesh objects, and stores them in a dictionary.
    
    This method assumes that each mesh has 3 separate data types, vertices, faces, and colors.
    """
   # Note: If load a .npz file, it becomes it's own class.
   # This class is similar to a dictionary, (e.g. loaded_data[key]).
    try:
        loaded_data = np.load(file) # lazy loading
    except FileNotFoundError:
        print(f"Error: File not found at path: {file}")
        return {}

    # Calculate the number of meshes from .npz file
    total_arrays = len(loaded_data.files)
    num_meshes = total_arrays // 3  # again, assume each mesh has 3 arrays: vertices, faces, colors
    mol_ids = range(1, num_meshes + 1)
    
    # Dictionary to hold the reconstructed meshes
    mol_meshes = {}
    desc = f"Loading {num_meshes} meshes with 1 cores"
    
    # This for loops iterates through each mesh ID, skipping if that ID is missing
    for mol_id in tqdm(mol_ids, desc=desc, total=num_meshes, colour='#7BC8F6'):
        key_prefix = f'mesh_{mol_id:04d}'
        try:
            loaded_vertices = loaded_data[f'{key_prefix}_vertices']
            loaded_faces = loaded_data[f'{key_prefix}_faces']
            loaded_colors = loaded_data[f'{key_prefix}_colors']
        except KeyError as e: # handles missing mesh IDs
            print(f"Error: Missing mesh_{mol_id:04d}. Skipping this molecule.")
            continue
            
        # Reconstruct the trimesh object
        reconstructed_mesh = trimesh.Trimesh(
            vertices=loaded_vertices,
            faces=loaded_faces,
            face_colors=loaded_colors # Note: face_colors is used for coloring faces
        )
        # Store the mesh in the dictionary
        mol_meshes[mol_id] = reconstructed_mesh

    return mol_meshes

In [ ]:
import os
import trimesh
import multiprocessing as mp
from tqdm import tqdm

def ply_count(path): # just for progress bar
    """Count the total number of PLY files in a directory."""
    count = 0
    with os.scandir(path) as entries:
        for entry in entries:
            if entry.is_file() and entry.name.endswith('.ply'):
                count += 1
    return count

def ply_find(path):
    """Generator yielding the full path of all PLY files in a directory"""
    with os.scandir(path) as entries:
        for entry in entries:
            if entry.is_file() and entry.name.endswith('.ply'): 
                yield entry.path

def ply_load_single_mesh(mesh_path):
    """
    Worker function to load a single mesh.
    This function is designed to be called by a multiprocessing pool.
    """
    name = os.path.basename(mesh_path).rsplit('.', 1)[0] # e.g. 'mesh_0001'
    mol_id = int(name.split('_')[-1])  # e.g. '0001' -> 1
    mesh = trimesh.load(mesh_path, file_type='ply', process=True)
    return mol_id, mesh

def ply_load_meshes(directory, num_processes=None, context='spawn'):
    """
    Load all PLY files from a directory into a dictionary of trimesh objects
    using multiprocessing.
    """    
    # Determine the number of processes
    if num_processes is None: 
        num_processes = mp.cpu_count()
    
    # Prepare for multiprocessing
    total_files = ply_count(directory)
    meshes_path = ply_find(directory) # generator

    ctx = mp.get_context(context)
    with ctx.Pool(processes=num_processes) as pool:
        # Use imap_unordered for a lazy, memory-efficient map
        tqdm_iterator = tqdm(
            pool.imap(ply_load_single_mesh, meshes_path),
            total=total_files,
            desc=f'Loading {total_files} meshes with {num_processes} cores',
            colour='#7BC8F6'
        )
        
        # Iterate over the results from the pool and populate the dictionary
        mol_meshes = {mol_id: mesh for mol_id, mesh in tqdm_iterator}
            
    return mol_meshes

In [ ]:
def import_meshes(path, num_processes=None, context='spawn'):
    """
    Load mesh files from a specified path, supporting both .npz archives 
    and directories containing .ply files.

    This function acts as a dispatcher: it checks the path type (file or directory) 
    and file extension to determine the appropriate loading method.

    Parameters
    ----------
    path : str
        The path to the source data. This must be either:
        1. A file path ending in '.npz' (a compressed archive).
        2. A directory path containing one or more '.ply' mesh files.
    num_processes : int, optional
        The number of processes to use for parallel loading.
    context : str, optional
        The multiprocessing context to use ('spawn', 'fork', etc.).

    Returns
    -------
    mol_meshes : dict
        A dictionary where keys are molecule IDs (integers) and values are
        the corresponding mesh objects (e.g., trimesh.Trimesh instances).
    """
    # 1. Npz file
    if os.path.isfile(path) and path.endswith('.npz'):
        return npz_load_meshes(path)
    
    # 2 . Directory of PLY files
    elif os.path.isdir(path):
        # Check if there are any .ply files in the directory
        has_ply = any(f.endswith('.ply') for f in os.listdir(path)) # checks for at least 1 PLY file
        if has_ply: return ply_load_meshes(path, num_processes=num_processes, context=context)
        else: raise ValueError(f"No .ply files found in directory: {path}")
        
    # 3. Unsupported file type
    else:
        raise ValueError(f"Unsupported path or file type: {path}")


Loading 1501 meshes with 1 cores: 100%|██████████| 1501/1501 [00:17<00:00, 87.82it/s]
